# Solar Filament Segmentation 2026 — end-to-end pipeline

This notebook reproduces the submitted predictions from the weights stored in `weights/`. The method is described in `README.md` and `report/main.pdf`. Re-training is optional and shown at the end.

Requirements: `pip install -r requirements.txt`; the Kaggle data unzipped into `data/raw/MAGFiLO_1.0_Kaggle_2026/`.

In [ ]:
import sys, os, glob, json
os.chdir(os.path.dirname(os.path.abspath('__file__')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
sys.path.insert(0, 'src')
import numpy as np, pandas as pd, cv2, torch, matplotlib.pyplot as plt
from common import RAW, ROOT, load_image, panoptic_quality
from predict import load_models, predict_prob, extract_instances
PARAMS = json.load(open('weights/params.json'))   # post-processing parameters of the final submission
WEIGHTS = sorted(glob.glob('weights/*.pt'))
print(PARAMS, WEIGHTS, torch.cuda.is_available())

## 1. Data and targets
Training annotations are COCO polygons from 1–3 annotators per image. `src/prepare.py` rasterises them into per-annotator instance maps and three *consensus* targets: the soft filament mask (fraction of annotators), a normalised distance-to-boundary map and the contact line between touching filaments.

In [ ]:
if not os.path.exists('data/masks/touch'):
    !python src/prepare.py
splits = pd.read_csv('data/splits.csv'); print(splits.fold.value_counts().sort_index().to_dict(), 'files per fold'); splits.head()

In [ ]:
stem = splits[splits.fold == 0].stem.iloc[3]
img = load_image(f'{RAW}/train/train_images/{stem}.jpeg')
soft = cv2.imread(f'data/masks/soft/{stem}.png', 0); dist = cv2.imread(f'data/masks/dist/{stem}.png', 0)
fig, ax = plt.subplots(1, 3, figsize=(18, 6)); ax[0].imshow(img, cmap='gray'); ax[0].set_title(stem); ax[1].imshow(soft, cmap='magma'); ax[1].set_title('consensus mask'); ax[2].imshow(dist, cmap='magma'); ax[2].set_title('distance target')
for a in ax: a.axis('off')
plt.show()

## 2. Models and inference
U-Nets (ResNet34 and EfficientNet-B3 encoders, ImageNet-pretrained, one input channel, three output channels). Whole 2048×2048 frames are predicted with flip test-time augmentation and the models are averaged.

In [ ]:
models = load_models(WEIGHTS, 'cuda')
out = predict_prob(models, img, 'cuda', tta=True)
inst = extract_instances(out, PARAMS['method'], PARAMS['thr'], PARAMS['min_area'], PARAMS['aux_thr'])
fig, ax = plt.subplots(1, 3, figsize=(18, 6)); ax[0].imshow(out[0], cmap='magma'); ax[0].set_title('filament probability'); ax[1].imshow(out[2], cmap='magma', vmax=1); ax[1].set_title('contact-line probability'); ax[2].imshow(inst, cmap='nipy_spectral'); ax[2].set_title(f'{inst.max()} instances')
for a in ax: a.axis('off')
plt.show()

## 3. Validation (fold 0, grouped by observation date)
Panoptic Quality against every annotator set of each validation image. Note: the weights in `weights/` were validated on fold 0 only if their name ends in `_f0`; other folds saw these images during training, so this cell is an illustration unless you restrict `WEIGHTS` to `_f0` models.

In [ ]:
images_df = pd.DataFrame(json.load(open(f'{RAW}/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'))['images'])
va = splits[splits.fold == 0].stem.tolist()
f0 = [w for w in WEIGHTS if w.endswith('_f0.pt')] or WEIGHTS
m0 = load_models(f0, 'cuda')
pqs = []
for s in va:
    o = predict_prob(m0, load_image(f'{RAW}/train/train_images/{s}.jpeg'), 'cuda', tta=True)
    ins = extract_instances(o, PARAMS['method'], PARAMS['thr'], PARAMS['min_area'], PARAMS['aux_thr'])
    for iid in images_df[images_df.file_name == s + '.jpeg'].id:
        gt = cv2.imread(f'data/masks/inst/{iid}.png', cv2.IMREAD_UNCHANGED).astype(np.int32)
        pqs.append(panoptic_quality(gt, ins)[0])
print(f'fold-0 models {f0}: validation PQ {np.mean(pqs):.4f} ± {np.std(pqs, ddof=1)/np.sqrt(len(pqs)):.4f} over {len(pqs)} annotation sets')

## 4. Test predictions and submission file
Each instance becomes one COCO RLE row; components are disjoint by construction, which `src/check_submission.py` verifies together with RLE decodability and id uniqueness.

In [ ]:
w = ' '.join(WEIGHTS)
!python src/predict.py --weights {w} --tta --method {PARAMS['method']} --thr {PARAMS['thr']} --min_area {PARAMS['min_area']} --aux_thr {PARAMS['aux_thr']} --out submission.csv
!python src/check_submission.py submission.csv
pd.read_csv('submission.csv').head()

## 5. (Optional) Re-train and re-tune
```
scripts/train_folds.sh resnet34        resnet34d "0 1 2 3 4" --epochs 40 --eval_every 5
scripts/train_folds.sh efficientnet-b3 effb3d    "0 1 2 3 4" --epochs 50 --crop 640 --bs 6 --eval_every 10
python src/evaluate.py --weights runs/resnet34d_f0/best.pt runs/effb3d_f0/best.pt --tta
python src/export_weights.py runs/resnet34d_f0 weights/resnet34d_f0.pt
```